
# Topic 5 — Distillation + Model Compression

**Core question:** *How do I shrink a strong model and cut its cost?*

**Setup:** Teacher = `Qwen2.5-1.5B`, Student = `Qwen2.5-0.5B`. The teacher
generates high-quality outputs on customer-support prompts (reusing our
running example); the student is trained to match the teacher's behavior,
trading some capability for much lower inference cost.

```mermaid
flowchart LR
    P[Customer support prompts] --> T[Teacher: Qwen2.5-1.5B]
    T -->|generates high-quality responses| D[Distillation dataset]
    D --> S[Student: Qwen2.5-0.5B]
    S -->|learns to imitate the teacher's distribution| S2[Smaller, cheaper student model]
```

### Can you answer this?
1. What's a "soft target," and why is it more informative than a single correct label?
2. What does the temperature parameter actually do to the teacher's output distribution?
3. What's the difference between response-level and on-policy distillation?
4. What's the trade-off you're making by distilling into a smaller model?


In [1]:

from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich import box

console = Console()
def banner(title, subtitle="", style="bold white on blue"):
    text = title if not subtitle else f"{title}\n[dim]{subtitle}[/dim]"
    console.print(Panel(text, style=style, expand=False, box=box.DOUBLE))
def section(title):
    console.print(f"\n[bold cyan]── {title} " + "─" * max(0, 60 - len(title)) + "[/bold cyan]")
def kv_table(title, rows, caption=""):
    table = Table(title=title, box=box.SIMPLE_HEAVY, caption=caption)
    table.add_column("Key", style="bold")
    table.add_column("Value")
    for k, v in rows.items():
        table.add_row(str(k), str(v))
    console.print(table)

banner("Topic 5: Distillation + Model Compression", "Teacher Qwen2.5-1.5B -> Student Qwen2.5-0.5B")


╔══════════════════════════════════════════════╗
║ Topic 5: Distillation + Model Compression    ║
║ Teacher Qwen2.5-1.5B -> Student Qwen2.5-0.5B ║
╚══════════════════════════════════════════════╝


## Concept primer

**Teacher / student**: the teacher is a larger, more capable (and more
expensive to run) model. The student is a smaller model we want to make
behave as much like the teacher as possible, so we get *some* of the
teacher's quality at a much lower inference cost.

**Logits and soft targets**: a model's raw, pre-softmax output scores are
called logits. Turning them into a probability distribution (via softmax)
gives a "soft target" -- e.g. `{"16": 0.82, "61": 0.06, "26": 0.04, ...}` --
instead of a single hard label like `"16"`. The soft target carries extra
information: *how confident* the teacher was, and *what else* it considered
plausible. Training the student to match this whole distribution (not just
the top answer) is what makes distillation more informative than plain
supervised learning on the teacher's top answer alone.

**Temperature**: dividing logits by a temperature `T > 1` before softmax
"softens" the distribution -- it flattens out a very peaked distribution so
the smaller probabilities (what the teacher considered *second-most*
likely, etc.) become more visible to the student. `T = 1` is the normal
softmax; higher `T` reveals more of this "dark knowledge." We'll compute
this directly below.

**KL divergence**: `KL(teacher || student)` measures how different the
student's predicted distribution is from the teacher's. Minimizing it is
the (or a) loss term in distillation -- it pulls the student's distribution
toward the teacher's, not just toward the single correct token.

**Response-level vs. on-policy distillation**: response-level distillation
trains the student on a *fixed* set of teacher-generated outputs (like
ordinary SFT on teacher data). On-policy distillation (TRL's `GKDTrainer`
-- Generalized Knowledge Distillation) instead has the *student* generate
its own completions during training, and compares the student's
token-level distribution against the teacher's distribution on those same
completions. This exposes the student to its own mistakes during training,
which tends to generalize better than only ever seeing the teacher's
"perfect" outputs.

**Capacity vs. quality**: a smaller student simply has less representational
capacity than the teacher. Distillation narrows the quality gap versus
training the small model from scratch, but doesn't erase it -- the
trade-off is explicitly lower cost/latency for somewhat lower quality.


In [2]:

import torch
section("Environment check")
HAS_CUDA = torch.cuda.is_available()
kv_table("Runtime", {"CUDA available": HAS_CUDA, "PyTorch": torch.__version__})
if not HAS_CUDA:
    console.print(Panel(
        "[bold yellow]No CUDA GPU detected.[/bold yellow] Prompt loading, the KL-divergence/temperature "
        "worked example, and real parameter-count/cost comparisons below all run for real without a "
        "GPU. Actual teacher generation and student distillation training require a GPU -- run on "
        "Colab (T4) for those.",
        title="GPU NOT AVAILABLE HERE", style="yellow", box=box.HEAVY,
    ))


── Environment check ───────────────────────────────────────────

          Runtime          
                           
  Key              Value   
 ━━━━━━━━━━━━━━━━━━━━━━━━━ 
  CUDA available   False   
  PyTorch          2.14.1

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ GPU NOT AVAILABLE HERE ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ No CUDA GPU detected. Prompt loading, the KL-divergence/temperature worked example, and real                    ┃
┃ parameter-count/cost comparisons below all run for real without a GPU. Actual teacher generation and student    ┃
┃ distillation training require a GPU -- run on Colab (T4) for those.                                             ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

In [3]:

import sys
IN_COLAB = "google.colab" in sys.modules
packages = ["unsloth", "transformers", "trl", "peft", "bitsandbytes", "torch", "datasets", "rich"]
if IN_COLAB:
    get_ipython().system(f"pip install -q {' '.join(packages)}")
    console.print("[green]Installed packages for Colab.[/green]")
else:
    console.print("[dim]Not running in Colab -- assuming the project virtualenv already has these.[/dim]")


Not running in Colab -- assuming the project virtualenv already has these.

In [4]:
# --- Load the distillation prompt set ---------------------------------------
import json
import random
from pathlib import Path

section("Loading the distillation prompt set")

vsc_ipynb_file = globals().get("__vsc_ipynb_file__")
search_roots = [Path.cwd()]
if vsc_ipynb_file:
    search_roots.insert(0, Path(vsc_ipynb_file).resolve().parent)
search_roots += [root.parent for root in list(search_roots)]

RELATIVE_CANDIDATES = ["data/distillation_prompts.json", "05_distillation_compression/data/distillation_prompts.json"]
DATA_PATH = None
for root in search_roots:
    for rel in RELATIVE_CANDIDATES:
        candidate = root / rel
        if candidate.is_file():
            DATA_PATH = candidate
            break
    if DATA_PATH:
        break

if DATA_PATH is None:
    # Self-sufficient fallback: regenerate 200 prompts from the same
    # deterministic customer-support generator used in Topic 1, instead of
    # requiring Topic 1's output to be uploaded alongside this notebook.
    console.print("[dim]distillation_prompts.json not found nearby -- generating prompts in-memory "
                  "(reusing Topic 1's generator) instead of requiring a separate file upload.[/dim]")

    PRODUCTS = ["wireless earbuds", "coffee maker", "running shoes", "laptop stand",
                "desk lamp", "yoga mat", "bluetooth speaker", "phone case",
                "winter jacket", "kitchen blender", "backpack", "office chair",
                "water bottle", "gaming mouse", "electric toothbrush", "air fryer",
                "board game", "children's book", "garden hose", "car phone mount"]
    NAMES = ["Alex", "Jordan", "Priya", "Sam", "Taylor", "Morgan", "Chris",
             "Jamie", "Riley", "Casey", "Drew", "Avery"]

    def _order(): return f"#{random.randint(100000, 999999)}"
    def _days(lo=1, hi=14): return random.randint(lo, hi)

    def order_status():
        p, o, d = random.choice(PRODUCTS), _order(), _days(1, 10)
        return (f"I ordered a {p} with order number {o} {d} days ago and I still don't see any "
                f"shipping update. Can you tell me what's going on?",
                f"Thanks for reaching out about order {o}. I checked and your {p} is currently being "
                f"prepared at our fulfillment center and is on track to ship within 1-2 business days. "
                f"You'll receive a tracking email the moment it ships, and estimated delivery is "
                f"{_days(2,5)} days after that. I'll flag the order so it's monitored closely.")

    def returns_refunds():
        p, o = random.choice(PRODUCTS), _order()
        return (f"I want to return the {p} from order {o} because it's not what I expected. How do I "
                f"get a refund?",
                f"No problem at all -- I've started a return for the {p} in order {o}. You'll get a "
                f"prepaid return label by email within the hour. Once the item is scanned by the "
                f"carrier, your refund of the full purchase price will be issued to your original "
                f"payment method within 3-5 business days.")

    def damaged_item():
        p, o = random.choice(PRODUCTS), _order()
        return (f"The {p} I received in order {o} arrived damaged -- it looks like it was crushed in "
                f"shipping. What can you do?",
                f"I'm sorry the {p} arrived damaged, that's not the experience we want for you. I've "
                f"issued a free replacement for order {o} at no extra cost, and you don't need to "
                f"return the damaged item. The replacement will ship within 1 business day with "
                f"expedited shipping.")

    def payment_issue():
        o = _order()
        amount = random.choice([19.99, 29.99, 49.50, 89.00, 124.75])
        return (f"I was charged ${amount} twice for order {o}. Can you check my payment and fix this?",
                f"I checked order {o} and confirmed a duplicate charge of ${amount}. I've submitted a "
                f"refund for the extra charge, which will appear on your statement within 3-5 business "
                f"days. Only one charge of ${amount} will remain for your order.")

    def account_access():
        name = random.choice(NAMES)
        return (f"Hi, this is {name}. I can't log into my account, it keeps saying my password is "
                f"incorrect even after I reset it. Can you help?",
                f"Hi {name}, sorry for the trouble logging in. I've sent a fresh password reset link "
                f"to your account email -- it will be valid for 30 minutes. If it still doesn't work "
                f"after that, it may be a browser cache issue, so try an incognito window or clearing "
                f"cookies for our site.")

    def shipping_delay():
        p, o, d = random.choice(PRODUCTS), _order(), _days(5, 14)
        return (f"My {p} from order {o} was supposed to arrive {d} days ago and it still hasn't shown "
                f"up. This is really frustrating.",
                f"I completely understand the frustration, and I'm sorry for the delay on your {p} in "
                f"order {o}. I checked with the carrier and the package is delayed in transit but "
                f"still moving -- updated delivery is within 2 days. As an apology for the delay, I've "
                f"credited your account with a $10 voucher for your next order.")

    def cancellation():
        p, o = random.choice(PRODUCTS), _order()
        return (f"I need to cancel my order {o} for the {p} -- I accidentally ordered two of them. Is "
                f"it too late?",
                f"Good news -- order {o} hasn't shipped yet, so I was able to cancel it for you right "
                f"away. You will not be charged for this order, and any pending authorization will "
                f"drop off within a few business days. The other order for the {p} is unaffected.")

    def product_question():
        p = random.choice(PRODUCTS)
        return (f"Before I buy the {p}, can you tell me if it comes with a warranty and what the "
                f"return window is?",
                f"Great question! The {p} comes with a 1-year manufacturer warranty covering defects, "
                f"and you can return it within 30 days of delivery for any reason as long as it's in "
                f"like-new condition. If anything goes wrong after the return window, our support "
                f"team can still help with the manufacturer warranty.")

    def subscription_issue():
        return ("I signed up for a monthly subscription but I was charged again this month even "
                "though I thought I canceled it last week. Can you check?",
                "I looked into your subscription and I can confirm the cancellation request from "
                "last week did not fully process due to a billing sync issue on our end. I've now "
                "canceled the subscription effective immediately and refunded this month's charge "
                "in full -- it will reflect within 3-5 business days.")

    def promo_discount():
        code = random.choice(["SAVE10", "WELCOME15", "SUMMER20", "FIRST5"])
        return (f"I tried to use the promo code {code} at checkout but it says invalid. The order "
                f"total didn't get discounted. Can you help?",
                f"Thanks for flagging this. I checked the promo code {code} and applied the discount "
                f"manually to your most recent order, so you'll see the adjusted total reflected "
                f"shortly. In the future, that code works best when applied before adding a gift wrap "
                f"option, which was conflicting with it at checkout.")

    CATEGORIES = {
        "order_status": order_status, "returns_refunds": returns_refunds,
        "damaged_item": damaged_item, "payment_issue": payment_issue,
        "account_access": account_access, "shipping_delay": shipping_delay,
        "cancellation": cancellation, "product_question": product_question,
        "subscription_issue": subscription_issue, "promo_discount": promo_discount,
    }

    def generate_support_tickets(n=500, seed=42):
        rng_state = random.getstate()
        random.seed(seed)
        names = list(CATEGORIES.keys())
        per, rem = n // len(names), n % len(names)
        examples = []
        for i, name in enumerate(names):
            count = per + (1 if i < rem else 0)
            for _ in range(count):
                problem, response = CATEGORIES[name]()
                examples.append({"category": name, "problem": problem, "response": response})
        random.shuffle(examples)
        random.setstate(rng_state)
        return examples

    support_examples = generate_support_tickets()
    prompts = [ex["problem"] for ex in support_examples[:200]]
    DATA_PATH = Path.cwd() / "data" / "distillation_prompts.json"
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    with DATA_PATH.open("w") as f:
        json.dump(prompts, f, indent=2)
else:
    with DATA_PATH.open() as f:
        prompts = json.load(f)

console.print(f"Loaded [bold]{len(prompts)}[/bold] prompts from {DATA_PATH} (reused from Topic 1's running example)")
console.print(Panel(prompts[0], title="Sample prompt", box=box.ROUNDED))


── Loading the distillation prompt set ─────────────────────────

Loaded 200 prompts from 
/Users/bhanupraka.gopularam/aiprojects/llm-fine-tuning/05_distillation_compression/data/distillation_prompts.json 
(reused from Topic 1's running example)

╭───────────────────────────────────────────────── Sample prompt ─────────────────────────────────────────────────╮
│ I signed up for a monthly subscription but I was charged again this month even though I thought I canceled it   │
│ last week. Can you check?                                                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


## Temperature & KL divergence -- a real worked example

Let's make "soft targets" and "temperature" concrete with actual numbers,
using a small synthetic logit vector standing in for a model's output over
a tiny 5-token vocabulary. This needs no GPU and no real model -- just the
math.


In [5]:

# --- Worked example: temperature softening + KL divergence (CPU-safe, real) -
import torch.nn.functional as F

section("Worked example: how temperature changes a distribution")

VOCAB = ["16", "61", "26", "14", "40"]  # plausible numeric-answer tokens
teacher_logits = torch.tensor([4.0, 1.0, 0.5, 0.2, -1.0])  # teacher is quite confident in "16"

table = Table(title="Teacher's distribution at different temperatures", box=box.SIMPLE_HEAVY)
table.add_column("Token")
for T in [1.0, 2.0, 5.0]:
    table.add_column(f"T={T}")

rows_by_temp = {}
for T in [1.0, 2.0, 5.0]:
    probs = F.softmax(teacher_logits / T, dim=0)
    rows_by_temp[T] = probs

for i, token in enumerate(VOCAB):
    table.add_row(token, *[f"{rows_by_temp[T][i]:.3f}" for T in [1.0, 2.0, 5.0]])
console.print(table)
console.print(
    "\n[dim]At T=1, the teacher looks almost certain '16' is correct (the rest barely register). "
    "At T=5, the distribution flattens -- the student can now 'see' that the teacher also considered "
    "'61' and '26' somewhat plausible. This extra information is the 'dark knowledge' distillation "
    "tries to transfer.[/dim]"
)

section("Worked example: KL divergence between teacher and a partially-trained student")
student_logits_untrained = torch.tensor([0.1, 0.0, -0.1, 0.2, 0.0])  # basically guessing
student_logits_trained = torch.tensor([3.5, 1.2, 0.6, 0.1, -0.8])    # close to the teacher

teacher_probs = F.softmax(teacher_logits, dim=0)
for name, student_logits in [("Untrained student", student_logits_untrained), ("Trained student", student_logits_trained)]:
    student_log_probs = F.log_softmax(student_logits, dim=0)
    kl = F.kl_div(student_log_probs, teacher_probs, reduction="sum")
    console.print(f"{name}: KL(teacher || student) = [bold]{kl.item():.4f}[/bold]")
console.print("\n[dim]Lower KL = the student's distribution is closer to the teacher's. "
              "This is exactly the quantity distillation training drives down.[/dim]")


── Worked example: how temperature changes a distribution ──────

    Teacher's distribution at    
     different temperatures      
                                 
  Token   T=1.0   T=2.0   T=5.0  
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  16      0.902   0.614   0.347  
  61      0.045   0.137   0.190  
  26      0.027   0.107   0.172  
  14      0.020   0.092   0.162  
  40      0.006   0.050   0.128

At T=1, the teacher looks almost certain '16' is correct (the rest barely register). At T=5, the distribution 
flattens -- the student can now 'see' that the teacher also considered '61' and '26' somewhat plausible. This extra
information is the 'dark knowledge' distillation tries to transfer.

── Worked example: KL divergence between teacher and a partially-trained student 

Untrained student: KL(teacher || student) = 1.1227

Trained student: KL(teacher || student) = 0.0206

Lower KL = the student's distribution is closer to the teacher's. This is exactly the quantity distillation 
training drives down.


## Teacher and student parameter counts (real, computed)

Just like Topic 2, we build both models from their architecture configs --
no weight download needed -- to get real parameter counts instantly.


In [6]:

# --- Real parameter counts + cost/latency estimate --------------------------
from transformers import AutoConfig, AutoModelForCausalLM

TEACHER_NAME = "Qwen/Qwen2.5-1.5B"
STUDENT_NAME = "Qwen/Qwen2.5-0.5B"

section("Building teacher/student architectures (no weight download needed)")
teacher_cfg = AutoConfig.from_pretrained(TEACHER_NAME)
student_cfg = AutoConfig.from_pretrained(STUDENT_NAME)
teacher_model = AutoModelForCausalLM.from_config(teacher_cfg)
student_model = AutoModelForCausalLM.from_config(student_cfg)

teacher_params = sum(p.numel() for p in teacher_model.parameters())
student_params = sum(p.numel() for p in student_model.parameters())
ratio = teacher_params / student_params

kv_table("Real parameter counts", {
    "Teacher (Qwen2.5-1.5B)": f"{teacher_params:,}",
    "Student (Qwen2.5-0.5B)": f"{student_params:,}",
    "Teacher / student ratio": f"{ratio:.2f}x",
})

# A simple, clearly-labeled latency/cost model scaling roughly with parameter
# count -- not a measured benchmark, but concrete arithmetic from real counts.
BASE_LATENCY_MS_PER_TOKEN_PER_BILLION_PARAMS = 8.0  # illustrative constant
teacher_latency = BASE_LATENCY_MS_PER_TOKEN_PER_BILLION_PARAMS * (teacher_params / 1e9)
student_latency = BASE_LATENCY_MS_PER_TOKEN_PER_BILLION_PARAMS * (student_params / 1e9)

table = Table(title="Estimated per-token latency & relative cost (from real param counts)", box=box.SIMPLE_HEAVY)
table.add_column("Metric")
table.add_column("Teacher (1.5B)", style="red")
table.add_column("Student (0.5B)", style="green")
table.add_row("Parameters", f"{teacher_params:,}", f"{student_params:,}")
table.add_row("Est. latency / token", f"~{teacher_latency:.1f} ms", f"~{student_latency:.1f} ms")
table.add_row("Est. relative cost per request", "1.0x (baseline)", f"~{student_params/teacher_params:.2f}x")
console.print(table)
console.print("\n[dim]Latency estimate uses a simple linear-in-parameters illustrative constant, "
              "not a measured benchmark -- real serving latency also depends on batching, hardware, "
              "and KV-cache behavior. Measure real latency on Colab/production hardware for actual numbers.[/dim]")


── Building teacher/student architectures (no weight download needed) 

W1002 20:43:19.818000 33222 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


W1002 20:43:19.846000 33222 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


W1002 20:43:19.871000 33222 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


           Real parameter counts           
                                           
  Key                       Value          
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  Teacher (Qwen2.5-1.5B)    1,543,714,304  
  Student (Qwen2.5-0.5B)    494,032,768    
  Teacher / student ratio   3.12x

Estimated per-token latency & relative cost (from real param counts) 
                                                                     
  Metric                           Teacher (1.5B)    Student (0.5B)  
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  Parameters                       1,543,714,304     494,032,768     
  Est. latency / token             ~12.3 ms          ~4.0 ms         
  Est. relative cost per request   1.0x (baseline)   ~0.32x         

Latency estimate uses a simple linear-in-parameters illustrative constant, not a measured benchmark -- real serving
latency also depends on batching, hardware, and KV-cache behavior. Measure real latency on Colab/production 
hardware for actual numbers.


## Running the real distillation

TRL's `GKDTrainer` implements **on-policy distillation**: the student
generates its own completions during training, and is trained to match the
teacher's token-level distribution (via KL divergence, with the
temperature-scaled soft targets from above) on those completions.


In [7]:

# --- Real distillation run (GPU-only) ---------------------------------------
if HAS_CUDA:
    from unsloth import FastLanguageModel
    from trl import GKDConfig, GKDTrainer
    from datasets import Dataset

    teacher, teacher_tok = FastLanguageModel.from_pretrained(model_name=TEACHER_NAME, max_seq_length=512)
    student, student_tok = FastLanguageModel.from_pretrained(model_name=STUDENT_NAME, max_seq_length=512)

    SYSTEM_PROMPT = ("You are a helpful customer support assistant for an online shopping "
                      "platform. Respond to the customer's problem clearly and helpfully.")

    def to_gkd_example(prompt):
        conv = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": prompt}]
        return {"messages": conv}

    train_dataset = Dataset.from_list([to_gkd_example(p) for p in prompts])

    gkd_cfg = GKDConfig(output_dir="outputs/distillation", num_train_epochs=1,
                         per_device_train_batch_size=2, learning_rate=1e-5,
                         temperature=2.0, logging_steps=20, save_strategy="no", report_to=[])
    trainer = GKDTrainer(model=student, teacher_model=teacher, args=gkd_cfg,
                          train_dataset=train_dataset, processing_class=student_tok)
    section("Running on-policy distillation (student learns from teacher)")
    result = trainer.train()
    kv_table("Distillation result", {"final loss": result.training_loss})
else:
    console.print(Panel(
        "[yellow]SKIPPED[/yellow] -- loading the teacher+student models and running GKDTrainer requires a CUDA GPU.\n"
        "Expected on Colab (T4): the student's loss (KL divergence to the teacher's distribution)\n"
        "decreasing over training; qualitatively, the student's responses should start to pick up\n"
        "the teacher's tone/structure on customer-support prompts, even though it's 3x smaller.",
        title="On-policy distillation (GPU-only)", style="yellow",
    ))


╭─────────────────────────────────────── On-policy distillation (GPU-only) ───────────────────────────────────────╮
│ SKIPPED -- loading the teacher+student models and running GKDTrainer requires a CUDA GPU.                       │
│ Expected on Colab (T4): the student's loss (KL divergence to the teacher's distribution)                        │
│ decreasing over training; qualitatively, the student's responses should start to pick up                        │
│ the teacher's tone/structure on customer-support prompts, even though it's 3x smaller.                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


## Can you answer this? (Topic 5 training checkpoints)

**1. What's a "soft target," and why is it more informative than a single correct label?**
<details><summary>Answer</summary>
A soft target is the teacher's full probability distribution over possible
outputs/tokens, not just its top pick. It encodes relative confidence
across *all* options, which gives the student much richer signal about
*how* to think about the problem, not just *what* the final answer is.
</details>

**2. What does the temperature parameter actually do to the teacher's output distribution?**
<details><summary>Answer</summary>
Dividing logits by a temperature `T > 1` before softmax flattens the
distribution -- the gap between the top choice and the runner-up choices
shrinks, revealing more of what the teacher considered plausible but didn't
pick. At `T = 1` you get the normal, often very peaked distribution.
</details>

**3. What's the difference between response-level and on-policy distillation?**
<details><summary>Answer</summary>
Response-level distillation trains the student on a *fixed* set of
teacher-generated outputs -- essentially ordinary SFT using the teacher as
the data source. On-policy distillation (TRL's GKDTrainer) has the
*student* generate its own completions during training and compares those
against the teacher's distribution -- exposing the student to its own
mistakes, which tends to generalize better than only ever imitating
"perfect" teacher outputs.
</details>

**4. What's the trade-off you're making by distilling into a smaller model?**
<details><summary>Answer</summary>
Lower inference cost and latency (fewer parameters to run per token) in
exchange for some loss of quality/capability versus the teacher -- the
student has strictly less representational capacity. Distillation narrows
this gap versus training the small model from scratch on the same data,
but doesn't eliminate it.
</details>


In [8]:

section("Scale discussion: distillation at ~100 vs ~10k users")
table = Table(title="Distillation economics at scale", box=box.SIMPLE_HEAVY, show_lines=True)
table.add_column("Concern")
table.add_column("~100 users", style="cyan")
table.add_column("~10k users", style="magenta")
rows = [
    ("Is distillation worth it?", "Often not yet -- the teacher alone may be cheap enough at this volume",
                                   "Frequently yes -- running the teacher for every request at this volume gets expensive; a distilled student serving most traffic (with teacher fallback for hard cases) cuts cost substantially"),
    ("Teacher generation cost", "A few hundred teacher generations for training data is cheap and quick",
                                "Generating the training distillation set (10k+ prompts) becomes a real pipeline cost, often run as a batch job"),
    ("Serving cost per request", "Dominated by fixed overhead regardless of model size",
                                 "Scales roughly with parameter count -- the student's smaller footprint directly reduces GPU-hours and $/request at this volume"),
    ("Quality monitoring", "Spot-check student vs teacher outputs manually",
                            "Needs an ongoing eval pipeline comparing student quality against the teacher (or ground truth) to catch drift as traffic patterns evolve"),
]
for c, s, b in rows:
    table.add_row(c, s, b)
console.print(table)


── Scale discussion: distillation at ~100 vs ~10k users ────────

                                          Distillation economics at scale                                          
                                                                                                                   
  Concern                     ~100 users                                 ~10k users                                
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  Is distillation worth it?   Often not yet -- the teacher alone may     Frequently yes -- running the teacher     
                              be cheap enough at this volume             for every request at this volume gets     
                                                                         expensive; a distilled student serving    
                                                                         most traffic (with teacher fallback for   
                                                                         hard cases) cuts cost substantially       
                                                                                                                   
  Teacher generation cost     A few hundred teacher generations for      Generating the training distillation set  
                              training data is cheap and quick           (10k+ prompts) becomes a real pipeline    
                                                                         cost, often run as a batch job            
                                                                                                                   
  Serving cost per request    Dominated by fixed overhead regardless     Scales roughly with parameter count --    
                              of model size                              the student's smaller footprint directly  
                                                                         reduces GPU-hours and $/request at this   
                                                                         volume                                    
                                                                                                                   
  Quality monitoring          Spot-check student vs teacher outputs      Needs an ongoing eval pipeline comparing  
                              manually                                   student quality against the teacher (or   
                                                                         ground truth) to catch drift as traffic   
                                                                         patterns evolve                          

In [9]:

section("Summary")
console.print(Panel(
    "[bold]What we did:[/bold]\n"
    "  1. Reused Topic 1's customer-support prompts as the distillation prompt set\n"
    "  2. Worked a real numeric example of temperature softening and KL divergence\n"
    "  3. Built teacher (1.5B) and student (0.5B) architectures and computed REAL parameter counts\n"
    "  4. Estimated latency/cost from those real counts\n"
    "  5. (On GPU) ran on-policy distillation with TRL's GKDTrainer\n\n"
    "[bold]Next: Topic 6 -- Scaling, Evaluation & Production Fine-Tuning[/bold]\n"
    "We take the LoRA model from Topic 2 and build out training -> evaluation ->\n"
    "model registry -> serving -> monitoring, the full production path.",
    title="Topic 5 complete", style="bold green", box=box.DOUBLE,
))


── Summary ─────────────────────────────────────────────────────

╔═══════════════════════════════════════════════ Topic 5 complete ════════════════════════════════════════════════╗
║ What we did:                                                                                                    ║
║   1. Reused Topic 1's customer-support prompts as the distillation prompt set                                   ║
║   2. Worked a real numeric example of temperature softening and KL divergence                                   ║
║   3. Built teacher (1.5B) and student (0.5B) architectures and computed REAL parameter counts                   ║
║   4. Estimated latency/cost from those real counts                                                              ║
║   5. (On GPU) ran on-policy distillation with TRL's GKDTrainer                                                  ║
║                                                                                                                 ║
║ Next: Topic 6 -- Scaling, Evaluation & Production Fine-Tuning                                                   ║
║ We take the LoRA model from Topic 2 and build out training -> evaluation ->                                     ║
║ model registry -> serving -> monitoring, the full production path.                                              ║
╚═════════════════════════════════════════════════════════════════════════════════════════════════════════════════╝